In [1]:
import numpy as np
import sympy as sp
from deom import benchmark, convert, decompose_spe, complex_2_json, init_qmd

Nmode = 1
Nbath = 1
Nper = 8

theta=0.5

Omega = np.zeros(Nmode, dtype=complex)
Omega[0] = 1

V = np.zeros([Nmode, Nmode], dtype=complex)
V[0, 0] = 1

eta = 0.5
gams = 2
temp = 0.2
beta = 1 / temp
npsd = Nper - 1


w_sp, eta_sp, gamma_sp, beta_sp = sp.symbols(
    r"\omega, \eta, \gamma, \beta", real=True)

phixx_sp = 2 * eta_sp * gamma_sp / (gamma_sp - sp.I * w_sp)
# phixx_sp = eta_sp * gamma_sp / (gamma_sp - sp.I * w_sp)
spe_vib_sp = phixx_sp
sp_para_dict = {eta_sp: eta, gamma_sp: gams}
condition_dict = {}
para_dict = {'beta': beta}
etal0, etar0, etaa0, expn0 = decompose_spe(spe_vib_sp, w_sp, sp_para_dict,
                                           para_dict, condition_dict, npsd)

expn = np.zeros([Nbath, Nper], dtype=complex)
expn[0, :] = expn0
etal = np.zeros([Nbath, Nmode, Nmode, Nper], dtype=complex)
etal[0, 0, 0, :] = etal0
etar = np.zeros([Nbath, Nmode, Nmode, Nper], dtype=complex)
etar[0, 0, 0, :] = etar0

etap = (etal + etar) / 2
etapp = (etal - etar) / 2 / 1j

In [2]:
def rem_1nd(meanx, meanp, theta1):
    dmeanx = np.zeros(Nmode, dtype=complex)
    dmeanp = np.zeros(Nmode, dtype=complex)
    dtheta1 = np.zeros([Nbath, Nmode, Nper], dtype=complex)

    for a in range(Nmode):
        dmeanx[a] = Omega[a] * meanp[a]
        for b in range(Nmode):
            dmeanp[a] -= V[a, b] * meanx[b]

        for alph in range(Nbath):
            for k in range(Nper):
                dmeanp[a] -= etap[alph, a, a, k] ** 0.5 * theta1[alph, a, k]
                dtheta1[alph, a, k] = -expn[alph, k] * theta1[alph, a, k]
                for b in range(Nmode):
                    dtheta1[alph, a, k] += 2 * etapp[alph, a, b, k] * meanx[b] / etap[alph, a, a, k] ** 0.5

    return dmeanx, dmeanp, dtheta1

def rem_2nd(sigmaxx, sigmaxp, sigmapp, Qdyn, Pdyn, thetadyn):
    dsigmaxx = np.zeros([Nmode, Nmode], dtype=complex)
    dsigmaxp = np.zeros([Nmode, Nmode], dtype=complex)
    dsigmapp = np.zeros([Nmode, Nmode], dtype=complex)
    dQdyn = np.zeros([Nmode, Nbath, Nmode, Nper], dtype=complex)
    dPdyn = np.zeros([Nmode, Nbath, Nmode, Nper], dtype=complex)
    dthetadyn = np.zeros([Nbath, Nmode, Nper, Nbath, Nmode, Nper], dtype=complex)

    for a in range(Nmode):
        for b in range(Nmode):
            dsigmaxx[a,
                     b] = Omega[a] * sigmaxp[b, a] + Omega[b] * sigmaxp[a, b]

            dsigmaxp[a, b] = Omega[a] * sigmapp[a, b]
            for ap in range(Nmode):
                dsigmaxp[a, b] -= V[ap, b] * sigmaxx[a, ap]
            for alph in range(Nbath):
                for k in range(Nper):
                    dsigmaxp[a, b] -= Qdyn[a, alph, b, k]*(etap[alph,b,b,k])**(0.5)

            dsigmapp[a, b] = 0
            for ap in range(Nmode):
                dsigmapp[
                    a,
                    b] -= V[a, ap] * sigmaxp[ap, b] + V[b, ap] * sigmaxp[ap, a]
            for alph in range(Nbath):
                for k in range(Nper):
                    dsigmapp[a, b] -= Pdyn[b, alph, a, k]*(etap[alph,a,a,k])**(0.5) + Pdyn[a, alph, b, k]*(etap[alph,b,b,k])**(0.5)

    for a in range(Nmode):
        for alph in range(Nbath):
            for ap in range(Nmode):
                for k in range(Nper):
                    dQdyn[a, alph, ap, k] = Omega[a] * Pdyn[
                        a, alph, ap, k] - expn[alph, k] * Qdyn[a, alph, ap, k]
                    for b in range(Nmode):
                        dQdyn[a, alph, ap,
                              k] += 2 * etapp[alph, ap, b, k]/(etap[alph,ap,ap,k])**(0.5) * sigmaxx[a, b]

                    dPdyn[a, alph, ap,
                          k] = -etap[alph, ap, a,
                                     k]/(etap[alph,ap,ap,k])**(0.5) - expn[alph, k] * Pdyn[a, alph, ap, k]
                    for b in range(Nmode):
                        dPdyn[
                            a, alph, ap,
                            k] += -V[a, b] * Qdyn[b, alph, ap, k] + 2 * etapp[
                                alph, ap, b, k] * sigmaxp[b, a]/(etap[alph,ap,ap,k])**(0.5)
                    for alphp in range(Nbath):
                        for kp in range(Nper):
                            dPdyn[a, alph, ap, k] -= thetadyn[alph, ap, k,
                                                              alphp, a, kp]*(etap[alphp,a,a,kp])**(0.5)

    for alph in range(Nbath):
        for a in range(Nmode):
            for k in range(Nper):
                for alphp in range(Nbath):
                    for b in range(Nmode):
                        for j in range(Nper):
                            dthetadyn[alph, a, k, alphp, b,
                                      j] = -(expn[alph, k] + expn[alphp, j]
                                             ) * thetadyn[alph, a, k, alphp, b,
                                                          j]
                            for bp in range(Nmode):
                                dthetadyn[
                                    alph, a, k, alphp, b,
                                    j] += 2 * etapp[alph, a, bp, k]/(etap[alph,a,a,k])**(0.5) * Qdyn[
                                        bp, alphp, b, j] + 2 * etapp[
                                            alphp, b, bp, j]/(etap[alphp,b,b,j])**(0.5) * Qdyn[bp, alph,
                                                                    a, k]

    return dsigmaxx, dsigmaxp, dsigmapp, dQdyn, dPdyn, dthetadyn

meanx = np.zeros(Nmode, dtype=complex)
meanp = np.zeros(Nmode, dtype=complex)
theta1 = np.zeros([Nbath, Nmode, Nper], dtype=complex)

meanx[0] = 0.
meanp[0] = 0.1

sigmaxx = np.zeros([Nmode, Nmode], dtype=complex)
sigmaxp = np.zeros([Nmode, Nmode], dtype=complex)
sigmapp = np.zeros([Nmode, Nmode], dtype=complex)
Qdyn = np.zeros([Nmode, Nbath, Nmode, Nper], dtype=complex)
Pdyn = np.zeros([Nmode, Nbath, Nmode, Nper], dtype=complex)
thetadyn = np.zeros([Nbath, Nmode, Nper, Nbath, Nmode, Nper], dtype=complex)

sigmaxx[0, 0] = 0.5 + meanx[0] ** 2
sigmapp[0, 0] = 0.5 + meanp[0] ** 2


Nt = int(1/2)
dt = 0.01

t = 0.0

e_x = []
e_p = []
var_x = []
var_p = []
var_xp = []
t_l = []

for i in range(Nt):
    t_l.append(t)
    e_x.append(meanx[0].real)
    e_p.append(meanp[0].real)
    var_x.append(sigmaxx[0, 0])
    var_p.append(sigmapp[0, 0])
    var_xp.append(sigmaxp[0, 0])
    
    # rk4 
    J1a, J1b, J1c = rem_1nd(meanx, meanp, theta1)
    meanx1, meanp1, theta11 = J1a, J1b, J1c

    J2a, J2b, J2c = rem_1nd(meanx + dt * meanx1 / 2, meanp + dt * meanp1 / 2, theta1 + dt * theta11 / 2)
    meanx2, meanp2, theta12 = J2a, J2b, J2c

    J3a, J3b, J3c = rem_1nd(meanx + dt * meanx2 / 2, meanp + dt * meanp2 / 2, theta1 + dt * theta12 / 2)
    meanx3, meanp3, theta13 = J3a, J3b, J3c

    J4a, J4b, J4c = rem_1nd(meanx + dt * meanx3, meanp + dt * meanp3, theta1 + dt * theta13)
    meanx4, meanp4, theta14 = J4a, J4b, J4c

    meanx += (meanx1 + 2*meanx2 + 2*meanx3 + meanx4)/6 * dt
    meanp += (meanp1 + 2*meanp2 + 2*meanp3 + meanp4)/6 * dt
    theta1 += (theta11 + 2*theta12 + 2*theta13 + theta14)/6 * dt

    K1a, K1b, K1c, K1d, K1e, K1f = rem_2nd(sigmaxx, sigmaxp, sigmapp, Qdyn, Pdyn, thetadyn)
    sigmaxx1, sigmaxp1, sigmapp1, Qdyn1, Pdyn1, thetadyn1 = K1a, K1b, K1c, K1d, K1e, K1f

    K2a, K2b, K2c, K2d, K2e, K2f = rem_2nd(sigmaxx + dt * sigmaxx1 / 2, sigmaxp + dt * sigmaxp1 / 2, sigmapp + dt * sigmapp1 / 2, Qdyn + dt * Qdyn1 / 2, Pdyn + dt * Pdyn1 / 2, thetadyn + dt * thetadyn1 / 2) 
    sigmaxx2, sigmaxp2, sigmapp2, Qdyn2, Pdyn2, thetadyn2 = K2a, K2b, K2c, K2d, K2e, K2f

    K3a, K3b, K3c, K3d, K3e, K3f = rem_2nd(sigmaxx + dt * sigmaxx2 / 2, sigmaxp + dt * sigmaxp2 / 2, sigmapp + dt * sigmapp2 / 2, Qdyn + dt * Qdyn2 / 2, Pdyn + dt * Pdyn2 / 2, thetadyn + dt * thetadyn2 / 2)
    sigmaxx3, sigmaxp3, sigmapp3, Qdyn3, Pdyn3, thetadyn3 = K3a, K3b, K3c, K3d, K3e, K3f
    
    K4a, K4b, K4c, K4d, K4e, K4f = rem_2nd(sigmaxx + dt * sigmaxx3, sigmaxp + dt * sigmaxp3, sigmapp + dt * sigmapp3, Qdyn + dt * Qdyn3, Pdyn + dt * Pdyn3, thetadyn + dt * thetadyn3)
    sigmaxx4, sigmaxp4, sigmapp4, Qdyn4, Pdyn4, thetadyn4 = K4a, K4b, K4c, K4d, K4e, K4f

    sigmaxx += (sigmaxx1 + 2*sigmaxx2 + 2*sigmaxx3 + sigmaxx4)/6 * dt
    sigmaxp += (sigmaxp1 + 2*sigmaxp2 + 2*sigmaxp3 + sigmaxp4)/6 * dt
    sigmapp += (sigmapp1 + 2*sigmapp2 + 2*sigmapp3 + sigmapp4)/6 * dt
    Qdyn += (Qdyn1 + 2*Qdyn2 + 2*Qdyn3 + Qdyn4)/6 * dt
    Pdyn += (Pdyn1 + 2*Pdyn2 + 2*Pdyn3 + Pdyn4)/6 * dt
    thetadyn += (thetadyn1 + 2*thetadyn2 + 2*thetadyn3 + thetadyn4)/6 * dt

    t += dt
    
#     str_ = ""
#     for a in range(Nmode):
#         for b in range(Nmode):
#             str_ += "{:16.6e}".format(np.real(sigmapp[a,b]))
    
#     for a in range(Nmode):
#         for b in range(Nmode):
#             str_ += "{:16.6e}".format(np.real(sigmaxp[a,b]))
#     print(str_)
    
#     tt_out = np.append(tt_out,t)
#     outputL = np.append(outputL, currentL.real)
#     outputR = np.append(outputR, currentR.real)
    
#     WQQ00_out = np.append(WQQ00_out,sigmaxx[0,0].real)
#     WQQ01_out = np.append(WQQ01_out,sigmaxx[0,1].real)
#     WQQ10_out = np.append(WQQ10_out,sigmaxx[1,0].real)
#     WQQ11_out = np.append(WQQ11_out,sigmaxx[1,1].real)

#     WQP00_out = np.append(WQP00_out,sigmaxp[0,0].real)
#     WQP01_out = np.append(WQP01_out,sigmaxp[0,1].real)
#     WQP10_out = np.append(WQP10_out,sigmaxp[1,0].real)
#     WQP11_out = np.append(WQP11_out,sigmaxp[1,1].real)

#     WPP00_out = np.append(WPP00_out,sigmapp[0,0].real)
#     WPP01_out = np.append(WPP01_out,sigmapp[0,1].real)
#     WPP10_out = np.append(WPP10_out,sigmapp[1,0].real)
#     WPP11_out = np.append(WPP11_out,sigmapp[1,1].real)
    
# np.savetxt('second-DEOM-QQ.dat', np.column_stack([tt_out,WQQ00_out,WQQ01_out,WQQ10_out,WQQ10_out]))
# np.savetxt('second-DEOM-QP.dat', np.column_stack([tt_out,WQP00_out,WQP01_out,WQP10_out,WQP10_out]))
# np.savetxt('second-DEOM-PP.dat', np.column_stack([tt_out,WPP00_out,WPP01_out,WPP10_out,WPP10_out]))

In [ ]:
np.savetxt('time.dat', t_l)
np.savetxt('meanx.dat_{}'.format(eta), e_x)
np.savetxt('meanp.dat_{}'.format(eta), e_p)
np.savetxt('varx.dat_{}'.format(eta), var_x)
np.savetxt('varp.dat_{}'.format(eta), var_p)
np.savetxt('varxp.dat_{}'.format(eta), var_xp)

In [ ]:
e_x = np.array(e_x)
e_p = np.array(e_p)

plt.plot(t_l, e_x)
# plt.xlim(0, 20)
# plt.ylim(0, 5)
plt.show()
plt.plot(t_l, e_p)